# 00_configuracion_databricks

Configuración común del proyecto TCGA/RNA-Seq en Databricks.

Este archivo conserva la estructura funcional del `00_configuracion` probado en Databricks y agrega
las funciones reutilizables que necesitan los notebooks 02 a 07.

In [0]:
from pathlib import Path
import os, re, shutil, warnings, gc
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

In [0]:
# Arquitectura batch en Databricks
# Se conserva la ruta base que ya funcionó en el entorno del proyecto.
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "tcga_cancer_ml"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
BASE_DIR = Path(BASE_PATH)

RAW_PATH = BASE_DIR / "raw"
TRUSTED_PATH = BASE_DIR / "trusted"
REFINED_PATH = BASE_DIR / "refined"
MODELS_PATH = BASE_DIR / "models"

RAW_RNASEQ_PATH = RAW_PATH / "rnaseq"
RAW_METADATA_PATH = RAW_PATH / "metadata"

TRUSTED_LONG_PATH = TRUSTED_PATH / "rnaseq_long"
TRUSTED_MATRIX_PATH = TRUSTED_PATH / "rnaseq_matrix"

# Rutas adicionales requeridas por los notebooks 02 a 07
TRUSTED_SAMPLES_PATH = TRUSTED_PATH / "samples_18_clases"
TRUSTED_GENES_PATH = TRUSTED_PATH / "gene_dictionary"

REFINED_EDA_PATH = REFINED_PATH / "eda_outputs"
REFINED_METRICS_PATH = REFINED_PATH / "model_metrics"
REFINED_PREDICTIONS_PATH = REFINED_PATH / "predictions"
REFINED_VISUALIZATIONS_PATH = REFINED_PATH / "visualizations"

# Rutas adicionales para tablas, CSV y prototipo de aplicación
REFINED_TABLES_PATH = REFINED_PATH / "tables"
REFINED_EXPORTS_PATH = REFINED_PATH / "exports"
REFINED_APP_EXPORTS_PATH = REFINED_PATH / "app_exports"

# Parámetros generales del proyecto
SEED = 42
XGB_NUM_WORKERS = int(os.getenv("TCGA_XGB_NUM_WORKERS", "1"))
CV_PARALLELISM = int(os.getenv("TCGA_CV_PARALLELISM", "2"))

CLASES_PRINCIPALES = [
    "BRCA", "KIRC", "LUAD", "UCEC", "THCA", "HNSC",
    "LUSC", "PRAD", "LGG", "COAD", "SKCM", "STAD",
    "OV", "BLCA", "LIHC", "GBM", "KIRP", "CESC"
]

PROYECTOS_PRINCIPALES = [
    "TCGA-BRCA", "TCGA-KIRC", "TCGA-LUAD", "TCGA-UCEC",
    "TCGA-THCA", "TCGA-HNSC", "TCGA-LUSC", "TCGA-PRAD",
    "TCGA-LGG", "TCGA-COAD", "TCGA-SKCM", "TCGA-STAD",
    "TCGA-OV", "TCGA-BLCA", "TCGA-LIHC", "TCGA-GBM",
    "TCGA-KIRP", "TCGA-CESC"
]

mapa_cancer = {
    "TCGA-BRCA": "BRCA",
    "TCGA-KIRC": "KIRC",
    "TCGA-LUAD": "LUAD",
    "TCGA-UCEC": "UCEC",
    "TCGA-THCA": "THCA",
    "TCGA-HNSC": "HNSC",
    "TCGA-LUSC": "LUSC",
    "TCGA-PRAD": "PRAD",
    "TCGA-LGG": "LGG",
    "TCGA-COAD": "COAD",
    "TCGA-SKCM": "SKCM",
    "TCGA-STAD": "STAD",
    "TCGA-OV": "OV",
    "TCGA-BLCA": "BLCA",
    "TCGA-LIHC": "LIHC",
    "TCGA-GBM": "GBM",
    "TCGA-KIRP": "KIRP",
    "TCGA-CESC": "CESC"
}

mapa_nombre_cancer = {
    "TCGA-BRCA": "Breast invasive carcinoma",
    "TCGA-KIRC": "Kidney renal clear cell carcinoma",
    "TCGA-LUAD": "Lung adenocarcinoma",
    "TCGA-UCEC": "Uterine corpus endometrial carcinoma",
    "TCGA-THCA": "Thyroid carcinoma",
    "TCGA-HNSC": "Head and neck squamous cell carcinoma",
    "TCGA-LUSC": "Lung squamous cell carcinoma",
    "TCGA-PRAD": "Prostate adenocarcinoma",
    "TCGA-LGG": "Brain lower grade glioma",
    "TCGA-COAD": "Colon adenocarcinoma",
    "TCGA-SKCM": "Skin cutaneous melanoma",
    "TCGA-STAD": "Stomach adenocarcinoma",
    "TCGA-OV": "Ovarian serous cystadenocarcinoma",
    "TCGA-BLCA": "Bladder urothelial carcinoma",
    "TCGA-LIHC": "Liver hepatocellular carcinoma",
    "TCGA-GBM": "Glioblastoma multiforme",
    "TCGA-KIRP": "Kidney renal papillary cell carcinoma",
    "TCGA-CESC": "Cervical squamous cell carcinoma and endocervical adenocarcinoma"
}

print("Configuración cargada correctamente.")
print("Ruta base:", BASE_PATH)
print("Número de clases oficiales:", len(CLASES_PRINCIPALES))

Configuración cargada correctamente.
Ruta base: /Volumes/workspace/default/tcga_cancer_ml
Número de clases oficiales: 18


In [0]:
# Crear estructura del datalake batch
try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
except Exception as e:
    print(f"Aviso: no se pudo crear el volumen {CATALOG}.{SCHEMA}.{VOLUME}. Si ya existe, ignore este mensaje.")
    print("Detalle:", e)

rutas = [
    BASE_DIR,
    RAW_PATH,
    TRUSTED_PATH,
    REFINED_PATH,
    MODELS_PATH,
    RAW_RNASEQ_PATH,
    RAW_METADATA_PATH,
    TRUSTED_LONG_PATH,
    TRUSTED_MATRIX_PATH,
    TRUSTED_SAMPLES_PATH,
    TRUSTED_GENES_PATH,
    REFINED_EDA_PATH,
    REFINED_METRICS_PATH,
    REFINED_PREDICTIONS_PATH,
    REFINED_VISUALIZATIONS_PATH,
    REFINED_TABLES_PATH,
    REFINED_EXPORTS_PATH,
    REFINED_APP_EXPORTS_PATH
]

for ruta in rutas:
    dbutils.fs.mkdirs(str(ruta))

print("Estructura creada correctamente:")
for ruta in rutas:
    print(str(ruta))

Estructura creada correctamente:
/Volumes/workspace/default/tcga_cancer_ml
/Volumes/workspace/default/tcga_cancer_ml/raw
/Volumes/workspace/default/tcga_cancer_ml/trusted
/Volumes/workspace/default/tcga_cancer_ml/refined
/Volumes/workspace/default/tcga_cancer_ml/models
/Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq
/Volumes/workspace/default/tcga_cancer_ml/raw/metadata
/Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_long
/Volumes/workspace/default/tcga_cancer_ml/trusted/rnaseq_matrix
/Volumes/workspace/default/tcga_cancer_ml/trusted/samples_18_clases
/Volumes/workspace/default/tcga_cancer_ml/trusted/gene_dictionary
/Volumes/workspace/default/tcga_cancer_ml/refined/eda_outputs
/Volumes/workspace/default/tcga_cancer_ml/refined/model_metrics
/Volumes/workspace/default/tcga_cancer_ml/refined/predictions
/Volumes/workspace/default/tcga_cancer_ml/refined/visualizations
/Volumes/workspace/default/tcga_cancer_ml/refined/tables
/Volumes/workspace/default/tcga_cancer_ml/refined/ex

In [0]:
# Funciones auxiliares de rutas, tablas y persistencia

def path_str(path):
    """Convierte rutas Path o string en string compatible con Spark/dbutils."""
    return str(path)


def _safe_table_name(nombre_tabla: str) -> str:
    """Estandariza nombres de tabla para Unity Catalog."""
    return re.sub(r"[^A-Za-z0-9_]", "_", str(nombre_tabla))


def full_table_name(nombre_tabla: str) -> str:
    """Nombre completo de tabla en Unity Catalog."""
    nombre_tabla = _safe_table_name(nombre_tabla)
    return f"`{CATALOG}`.`{SCHEMA}`.`{nombre_tabla}`"


def table_path(nombre_tabla: str):
    """Ruta física Delta dentro del Volume para una tabla refined."""
    return REFINED_TABLES_PATH / _safe_table_name(nombre_tabla)


def _rm_path(path):
    """Elimina una ruta en DBFS/Volumes si existe."""
    try:
        dbutils.fs.rm(str(path), recurse=True)
    except Exception:
        pass


def table_exists_databricks(nombre_tabla: str) -> bool:
    """Valida si una tabla existe en Unity Catalog o como copia Delta en Volumes."""
    nombre_tabla = _safe_table_name(nombre_tabla)
    try:
        spark.table(full_table_name(nombre_tabla)).limit(1).count()
        return True
    except Exception:
        try:
            dbutils.fs.ls(str(table_path(nombre_tabla)))
            return True
        except Exception:
            return False

# Alias por compatibilidad con notebooks previos que usaban otra capitalización.
table_exists_Databricks = table_exists_databricks


def first_existing(candidates):
    """Retorna la primera ruta existente dentro de una lista de candidatos."""
    for c in candidates:
        if c is None:
            continue
        c = Path(str(c))
        try:
            dbutils.fs.ls(str(c))
            return c
        except Exception:
            try:
                if c.exists():
                    return c
            except Exception:
                pass
    return None


RAW_METADATA_FILE = first_existing([
    RAW_METADATA_PATH / "metadatos_tcga_oficial_18_clases.csv",
    RAW_PATH / "metadatos_tcga_oficial_18_clases.csv",
])

# Permite sobreescribir la ruta por variable de entorno si fuera necesario.
RAW_RNASEQ_PATH = (
    Path(os.getenv("TCGA_RAW_RNASEQ_DIR"))
    if os.getenv("TCGA_RAW_RNASEQ_DIR")
    else first_existing([RAW_PATH / "rnaseq", RAW_PATH / "RNASeq", RAW_PATH / "rna_seq", RAW_PATH])
)

print("RAW_METADATA_FILE:", RAW_METADATA_FILE)
print("RAW_RNASEQ_PATH:", RAW_RNASEQ_PATH)

RAW_METADATA_FILE: /Volumes/workspace/default/tcga_cancer_ml/raw/metadata/metadatos_tcga_oficial_18_clases.csv
RAW_RNASEQ_PATH: /Volumes/workspace/default/tcga_cancer_ml/raw/rnaseq


In [0]:
def mostrar(obj, n=10):
    """Muestra Spark DataFrames o pandas DataFrames de forma consistente en Databricks."""
    if hasattr(obj, "limit"):
        return display(obj.limit(n))
    return display(obj)


def save_spark_table(df, nombre_tabla, partition_by=None, export_csv=True, csv_max_rows=200000):
    """
    Guarda una tabla como Delta table en Unity Catalog y deja copia física en Volumes.

    Uso principal: salidas refined de EDA, feature selection, métricas, predicciones,
    reportes de clasificación, app_exports y negocio.
    """
    nombre_tabla = _safe_table_name(nombre_tabla)
    ruta = table_path(nombre_tabla)

    _rm_path(ruta)

    writer_path = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    writer_table = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )

    if partition_by:
        writer_path = writer_path.partitionBy(*partition_by)
        writer_table = writer_table.partitionBy(*partition_by)

    writer_path.save(str(ruta))
    writer_table.saveAsTable(full_table_name(nombre_tabla))

    if export_csv:
        try:
            n = df.count()
            if n <= csv_max_rows:
                dbutils.fs.mkdirs(str(REFINED_EXPORTS_PATH))
                df.toPandas().to_csv(REFINED_EXPORTS_PATH / f"{nombre_tabla}.csv", index=False)
        except Exception as e:
            print(f"No se exportó CSV para {nombre_tabla}: {e}")

    print("Tabla Delta guardada:", full_table_name(nombre_tabla))
    print("Copia física en Volume:", str(ruta))
    return ruta


def load_spark_table(nombre_tabla):
    """Carga primero desde Unity Catalog; si no existe, intenta leer Delta desde Volumes."""
    nombre_tabla = _safe_table_name(nombre_tabla)
    try:
        return spark.table(full_table_name(nombre_tabla))
    except Exception:
        ruta = table_path(nombre_tabla)
        try:
            return spark.read.format("delta").load(str(ruta))
        except Exception as e:
            raise FileNotFoundError(f"No existe la tabla {full_table_name(nombre_tabla)} ni la ruta {ruta}") from e


def _trusted_table_name_from_path(path):
    p = str(Path(str(path)))
    mapping = {
        str(TRUSTED_LONG_PATH): "trusted_tcga_rnaseq_long_18_clases",
        str(TRUSTED_SAMPLES_PATH): "trusted_tcga_samples_18_clases",
        str(TRUSTED_GENES_PATH): "trusted_tcga_gene_dictionary",
        str(TRUSTED_MATRIX_PATH): "trusted_tcga_rnaseq_matrix_18_clases",
    }
    return mapping.get(p)


def save_trusted(df, path, partition_by=None):
    """
    Guarda datasets trusted en Delta dentro del Volume.

    Además, cuando la ruta corresponde a los datasets principales, registra una tabla
    en Unity Catalog con el nombre esperado por las validaciones y notebooks posteriores.
    """
    path = Path(str(path))
    _rm_path(path)

    writer_path = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_by:
        writer_path = writer_path.partitionBy(*partition_by)

    writer_path.save(str(path))

    tabla = _trusted_table_name_from_path(path)
    if tabla:
        writer_table = (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
        )
        if partition_by:
            writer_table = writer_table.partitionBy(*partition_by)
        writer_table.saveAsTable(full_table_name(tabla))
        print("Tabla trusted registrada:", full_table_name(tabla))

    print("Dataset trusted Delta guardado:", str(path))
    return path


def read_trusted(path):
    """Lee un dataset trusted desde Delta. Si aplica, usa la tabla UC como respaldo."""
    path = Path(str(path))
    tabla = _trusted_table_name_from_path(path)

    try:
        return spark.read.format("delta").load(str(path))
    except Exception:
        if tabla:
            try:
                return spark.table(full_table_name(tabla))
            except Exception:
                pass
        raise FileNotFoundError(f"No existe el dataset trusted Delta: {path}")


def guardar_figura(nombre_archivo, dpi=300):
    """Guarda la figura activa de matplotlib en refined/visualizations."""
    import matplotlib.pyplot as plt
    dbutils.fs.mkdirs(str(REFINED_VISUALIZATIONS_PATH))
    ruta = REFINED_VISUALIZATIONS_PATH / nombre_archivo
    plt.savefig(str(ruta), dpi=dpi, bbox_inches="tight")
    print("Figura guardada:", str(ruta))
    return ruta

In [0]:
# Configuración Spark ligera
# Note: Arrow optimization is automatically managed in Spark Connect (Serverless)
spark.conf.set("spark.sql.shuffle.partitions", os.getenv("SPARK_SQL_SHUFFLE_PARTITIONS", "200"))

In [0]:
# Validación opcional de consistencia entre tabla long y tabla samples.
# Se deja protegida para que el notebook 00 no falle cuando se ejecuta antes de construir trusted.

def validar_consistencia_trusted():
    validacion_consistencia = spark.sql(f"""
        SELECT
            (SELECT COUNT(DISTINCT sample_id)
             FROM {CATALOG}.{SCHEMA}.trusted_tcga_rnaseq_long_18_clases) AS muestras_unicas_long,

            (SELECT COUNT(*)
             FROM {CATALOG}.{SCHEMA}.trusted_tcga_samples_18_clases) AS filas_samples,

            (SELECT COUNT(DISTINCT sample_id)
             FROM {CATALOG}.{SCHEMA}.trusted_tcga_samples_18_clases) AS muestras_unicas_samples,

            (SELECT COUNT(DISTINCT patient_id)
             FROM {CATALOG}.{SCHEMA}.trusted_tcga_rnaseq_long_18_clases) AS pacientes_unicos_long,

            (SELECT COUNT(DISTINCT gene_id_base)
             FROM {CATALOG}.{SCHEMA}.trusted_tcga_rnaseq_long_18_clases) AS genes_unicos_long
    """)
    display(validacion_consistencia)
    return validacion_consistencia

try:
    if (
        table_exists_databricks("trusted_tcga_rnaseq_long_18_clases")
        and table_exists_databricks("trusted_tcga_samples_18_clases")
    ):
        validar_consistencia_trusted()
    else:
        print("Validación trusted omitida: aún no existen las tablas trusted principales.")
except Exception as e:
    print("Validación trusted omitida por seguridad. Detalle:", e)

print("00_configuracion listo para notebooks 02 a 07.")

muestras_unicas_long,filas_samples,muestras_unicas_samples,pacientes_unicos_long,genes_unicos_long
8335,8335,8335,8283,19944


00_configuracion_databricks listo para notebooks 02 a 07.
